## EDA on the CSV - data/team_15_train.csv

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

In [ ]:
# Testing the dataset import - SUCCESS

df = pd.read_csv("../data/team_15_train.csv")
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

In [ ]:
# Plotting the histograms for all columns

features = ["battery_v","temp_c","rings_per_day","wifi_rssi_dbm","age_months"]
target = "days_left"

df.hist(bins=10, edgecolor="black")

# 2. Scatter plots: days_left vs each input feature
fig, axes = plt.subplots(nrows=3,ncols=2)
axes = axes.flatten()

for i, feature in enumerate(features):
    axes[i].scatter(df[feature], df[target], alpha=0.5)

    axes[i].set_xlabel(f"{feature}")
    axes[i].set_ylabel("days_left")
    axes[i].set_title(f"{feature} v/s days_left")


plt.tight_layout()
plt.show()

In [ ]:
# trying to investigate battery_life + temp's influence on days_left. Reason: individually, both look a little weak 
# (with temp having some outliers.) I want to check if, brought together, do they have a better correlation with days_left?

print(df[["battery_v", "temp_c", "days_left"]].corr(method="pearson"))
print(df[["battery_v", "temp_c", "days_left"]].corr(method="spearman"))

plt.scatter(
    df["battery_v"],
    df["temp_c"],
    c=df["days_left"],
    cmap="viridis",
    alpha=0.6
)

plt.xlabel("battery_v")
plt.ylabel("temp_c")
plt.colorbar(label="days_left")
plt.title("Battery Voltage vs Temperature")
plt.show()

In [ ]:
# trying to remove outliers. The above graphs didn't yield anything substantial, so trying to remove the obvious outliers observed in 
# the histogram. Making changes to the dataframe, keeping CSV untouched. clean_df is the hypothetical "clean" one, hoping that it yields 
# something better.

# Count suspicious readings
print("Battery:", (df["battery_v"] == 0).sum())
print("Temperature:", (df["temp_c"] == -127).sum())
print("WiFi:", (df["wifi_rssi_dbm"] == 0).sum())

# Remove invalid readings
clean_df = df[
    (df["battery_v"] != 0) &
    (df["temp_c"] != -127) &
    (df["wifi_rssi_dbm"] != 0)
].copy()

print("Original rows:", len(df))
print("Cleaned rows:", len(clean_df))
print("Removed rows:", len(df) - len(clean_df))

In [ ]:
# regenerating correlation-coefficients for the cleaned data
print(clean_df[["battery_v", "temp_c", "days_left"]].corr(method="pearson"))
print(clean_df[["battery_v", "temp_c", "days_left"]].corr(method="spearman"))

In [9]:
# calculating baseline mean, and MAE

X=clean_df.drop(columns=["days_left"])
y=clean_df["days_left"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
mean=y_train.mean()
y_pred=[mean] * len(y_val)
mae=mean_absolute_error(y_val, y_pred)

print("mean prediction:", mean)
print("MAE: ", mae)

mean prediction: 87.21829113924052
MAE:  55.98715797212633


In [ ]:
#Linear regression using LSM

# Convert dataframes to NumPy arrays
X_train_np = X_train.to_numpy()
X_val_np = X_val.to_numpy()

y_train_np = y_train.to_numpy()
y_val_np = y_val.to_numpy()

# Add intercept (bias) column
X_train_bias = np.column_stack([
    np.ones(len(X_train_np)), X_train_np
])

X_val_bias = np.column_stack([
    np.ones(len(X_val_np)), X_val_np
])

# Fit linear regression using least squares
weights, residuals, rank, singular_values = np.linalg.lstsq(
    X_train_bias, y_train_np, rcond=None
)

# Generate predictions
y_pred = X_val_bias @ weights

# Calculate MAE
linear_mae = mean_absolute_error(y_val_np, y_pred)

print("Linear Regression MAE:", linear_mae)
print("Mean Baseline MAE:", mae)

# Plot predicted vs actual values
plt.figure(figsize=(7, 5))
plt.scatter(y_val_np, y_pred, alpha=0.5)

# Ideal prediction line
min_val = min(y_val_np.min(), y_pred.min())
max_val = max(y_val_np.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")

plt.xlabel("Actual days_left")
plt.ylabel("Predicted days_left")
plt.title("Linear Regression: Predicted vs Actual")
plt.show()

## Observations with Linear Regression: The predictions arent tightly coupled around the linear line, suggesting a more complex data relationship
- Regression to the mean: At higher actual days_left values, predictions tend to underestimate. At lower values, some predictions overestimate.
- Prediction spread: The points aren't tightly concentrated around the ideal diagonal, indicating substantial residual errors.
- Possible nonlinearity: The model may not be capturing all the relationships between sensor readings and battery life.
- Negative predictions: Linear regression can predict physically impossible negative battery life because it has no output constraints.

## next steps:
- Try the neural network: can it capture the non-linearity?
- If it doesn't report a better MAE: can we somehow combine features (battery_v and temmp_c) which can together produce a better linear signal?

In [ ]:
# Trying neural network
